In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 76.8 MB/s eta 0:00:00


In [ ]:
!pip install -U FlagEmbedding

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 15.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 10.2 MB/s eta 0:0

In [ ]:
import faiss
import json
import numpy as np
import torch

In [ ]:
with open('/content/drive/My Drive/bible_rag/bible.json', encoding="utf-8") as f:
  bible = json.load(f)

In [ ]:
from FlagEmbedding import FlagModel

model = FlagModel('BAAI/bge-small-en-v1.5',
                  query_instruction_for_retrieval="Represent this sentence for searching relevant passages: ",
                  use_fp16=True)
model.model.to("cuda")

NameError: name 'FlagModel' is not defined

In [ ]:
# from sentence_transformers import SentenceTransformer

# Load the model
# model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")

In [ ]:
verses = [i["text"] for i in bible]

In [ ]:
ids = np.array([i["metadata"]["chunk_id"] for i in bible])

In [ ]:
document_embeddings = model.encode_corpus(verses)

In [ ]:
document_embeddings.dtype, document_embeddings[0].shape[0]

In [ ]:
norms = np.linalg.norm(document_embeddings, axis=1)
print("largest:", norms.max(), "smallest:", norms.min(), "std:", norms.std())

In [ ]:
# add index to embeddings
index = faiss.IndexFlatIP(document_embeddings[0].shape[0])
index.add(document_embeddings)

In [ ]:
# Save the index to a file
path = r"/content/drive/My Drive/bible_rag/bge-small-en-v1.5.faiss"
try:
    faiss.write_index(index, path)
    print("Index saved!")
except Exception as e:
    print("Error saving index:", e)